# Cloud TensorFlow Serving Test

Notebook ini menguji endpoint TensorFlow Serving yang berjalan di cloud Render. Request prediksi menggunakan serialized `tf.train.Example` yang di-encode ke base64, sesuai signature model TFX.

In [ ]:
import base64
import json

import requests
import tensorflow as tf

TF_SERVING_URL = "https://aqilaziz-wine-quality-tf-serving.onrender.com"
PROMETHEUS_URL = "https://aqilaziz-wine-quality-prometheus.onrender.com"

print("TF Serving:", TF_SERVING_URL)
print("Prometheus:", PROMETHEUS_URL)

In [ ]:
feature_values = {
    "fixed_acidity": 7.4,
    "volatile_acidity": 0.70,
    "citric_acid": 0.00,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "ph": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4,
}

feature = {
    name: tf.train.Feature(float_list=tf.train.FloatList(value=[value]))
    for name, value in feature_values.items()
}
example = tf.train.Example(features=tf.train.Features(feature=feature))
serialized = base64.b64encode(example.SerializeToString()).decode("utf-8")

payload = {"instances": [{"examples": {"b64": serialized}}]}
print(json.dumps(payload, indent=2)[:500] + "...")

In [ ]:
metadata_response = requests.get(
    f"{TF_SERVING_URL}/v1/models/wine-quality",
    timeout=60,
)
print("Metadata status:", metadata_response.status_code)
print(metadata_response.text)
metadata_response.raise_for_status()

In [ ]:
prediction_response = requests.post(
    f"{TF_SERVING_URL}/v1/models/wine-quality:predict",
    json=payload,
    timeout=60,
)
print("Prediction status:", prediction_response.status_code)
print(json.dumps(prediction_response.json(), indent=2))
prediction_response.raise_for_status()

In [ ]:
metrics_response = requests.get(
    f"{TF_SERVING_URL}/monitoring/prometheus/metrics",
    timeout=60,
)
print("Metrics status:", metrics_response.status_code)
print("tensorflow_serving_request_count" in metrics_response.text)
print(metrics_response.text[:1000])
metrics_response.raise_for_status()